# Ejemplo rápido de uso con fMRI

Este notebook muestra el flujo mínimo para cargar un fMRI 4D, aplicar el pipeline interno de la librería y obtener un grafo funcional basado en atlas.

## 1. Importaciones

Importamos la API pública de EEGraph y la configuración del atlas. En la versión actual, los atlas soportados son: `aal`, `schaefer_100`, `schaefer_200` y `schaefer_400`.

In [1]:
from pathlib import Path
import numpy as np
from eegraph.graph import Graph
from mrigraph.config import AtlasConfig

ModuleNotFoundError: No module named 'mrigraph'

## 2. Rutas de entrada y salida

Sustituye `RUTA_FMRI` por tu archivo NIfTI 4D (`.nii` o `.nii.gz`). Los auxiliares son opcionales en esta versión.

In [ ]:
RUTA_FMRI = Path("./datos/mi_fmri.nii.gz")
SALIDA = Path("./salidas")
SALIDA.mkdir(exist_ok=True)

## 3. Carga del fMRI

Con fMRI solo necesitas indicar la ruta principal y `modality="fmri"`. Si tuvieras archivos auxiliares, podrías pasarlos como `auxiliary_paths`, pero no son obligatorios para que el flujo funcione.

In [2]:
G = Graph()
G.load_data(
    path=str(RUTA_FMRI),
    modality="fmri",
    # auxiliary_paths=["./datos/confounds.csv", "./datos/events.tsv"]
)

print("Modalidad:", G.modality)
print("Metadata inicial disponible:", list(G.metadata.keys()))

NameError: name 'Graph' is not defined

## 4. Configurar el atlas

Aquí elegimos AAL, pero puedes cambiarlo por cualquiera de los atlas soportados por la librería.Son:

- atlas_name="aal"
- atlas_name="schaefer_100"
- atlas_name="schaefer_200"
- atlas_name="schaefer_400"

In [ ]:
atlas_config = AtlasConfig(atlas_name="aal")
# Alternativas válidas:
# atlas_config = AtlasConfig(atlas_name="schaefer_100")
# atlas_config = AtlasConfig(atlas_name="schaefer_200")
# atlas_config = AtlasConfig(atlas_name="schaefer_400")

## 5. Ejecutar el pipeline de modelado

Desde la API pública, todo se hace con una sola llamada a `modelate()`. Internamente, la librería encadena: preprocesado, denoising, transformación a ROI y conectividad.

In [ ]:
grafo_fmri, connectivity_matrix = G.modelate(
    window_size=1.0,
    connectivity="pearson",
    threshold=0.6,
    atlas_config=atlas_config,
)

print("Shape de la matriz de conectividad:", np.array(connectivity_matrix).shape)
print("Número de nodos:", grafo_fmri.number_of_nodes())
print("Número de aristas:", grafo_fmri.number_of_edges())

## 6. Inspeccionar metadatos útiles

Tras el modelado, `Graph.metadata` guarda trazabilidad del pipeline. En particular, el `transform_bundle` y el `connectivity_bundle` son muy útiles para depurar o documentar.

In [ ]:
transform_bundle = G.metadata["transform_bundle"]
connectivity_bundle = G.metadata["connectivity_bundle"]

print("Atlas usado:", transform_bundle.atlas_name)
print("Atlas resampleado:", transform_bundle.atlas_resampled)
print("Espacio de centroides:", transform_bundle.centroid_coordinate_space)
print("Número de ROIs:", transform_bundle.transform_metadata["num_rois"])
print("Primeras labels ROI:", transform_bundle.roi_labels[:10])

## 7. Visualización del grafo

La visualización usa las posiciones anatómicas derivadas del atlas y una codificación por color de la profundidad.

In [ ]:
G.visualize_html(
    grafo_fmri,
    SALIDA / "fmri_ejemplo",
    auto_open=False
)

print("Archivo generado:", SALIDA / "fmri_ejemplo_plot.html")

## 8. Qué te llevas de este ejemplo

Con fMRI, el flujo mínimo es:

1. crear `Graph()`
2. llamar a `load_data(..., modality="fmri")`
3. crear un `AtlasConfig` con uno de los atlas soportados
4. llamar a `modelate(...)`
5. inspeccionar `Graph.metadata` si quieres trazabilidad
6. visualizar el resultado con `visualize_html()` o `visualize_png()`